# Parse Work References

## Overview

This notebook parses raw reference data from works and resolves them to OpenAlex work IDs.

## Data Flow

```
locations_mapped.references (raw reference structs)
    |
    v
work_reference_lists (one row per filed list: its hash and its work)
    |
    v
work_references table (exploded, with resolved cited_work_id)
    |
    v
referenced_works table (aggregated array of resolved IDs per work)
    |
    v
CreateWorksEnriched (merges into openalex_works.referenced_works)
```

## Filing (oxjob #2054, #835 B3 + B4)

Each record's list (`native_id`, `native_id_namespace`, `provenance`) with at least one reference is filed once, and filed
again when its contents change or its record moves to another work, so its rows always sit under the work that owns the
record now. A list whose record is gone, or has no reference now, loses its rows. Matches no step can make again (the
2025 bulk parse's title matches) are put back on the new rows when the record still lists the same reference text.
Every matching step matches to live works only.

## Pipeline Position

This notebook reads directly from `locations_mapped` rather than `openalex_works_base`. 
This is a known architectural inconsistency - see `qa/issues/open/references-pipeline-refactor-2026-01/` for the planned refactor.

## Key Tables

- **Input:** `openalex.works.locations_mapped` - raw `references` array (doi, pmid, arxiv, title, authors, year, raw)
- **Output:** `openalex.works.work_references` - exploded references with resolved `cited_work_id`
- **Output:** `openalex.works.work_reference_lists` - one row per filed list (hash, work)
- **Output:** `openalex.works.referenced_works` - aggregated resolved IDs per `citing_work_id`

## Resolution Strategy

References are resolved to `cited_work_id` by matching against `work_id_map` in priority order:
1. DOI match (the reference's DOI field)
2. Text DOI / arXiv id match (oxjob #1972): the DOI field is empty and the reference text prints a DOI or an arXiv id
   (`parsed_doi`, filled at insert; arXiv ids as `10.48550/arxiv.<id>`, also matched to `work_id_map.arxiv`), with guards
   against issue-level records, page text and references whose title and author contradict the DOI's work
3. PMID match (where DOI is null in work_id_map to avoid erroneous matches)
4. Title/author match: not implemented (the cell labelled title_author repeats the PMID merge; oxjob #835 B9)

### Create `openalex.works.work_references` table (if not exists)

In [0]:
%sql
-- looks like a few fields have been added after - citing_work_id, cited_work_id, etc.
CREATE TABLE IF NOT EXISTS openalex.works.work_references 
CLUSTER BY AUTO
AS
  WITH references_exploded AS (
    SELECT 
      native_id, 
      native_id_namespace,
      work_id as citing_work_id,
      provenance,
      posexplode(references) as (ref_ind, ref)
    FROM openalex.works.locations_mapped
  )
  SELECT
    native_id, 
    native_id_namespace,
    citing_work_id,
    ref_ind,
    ref.doi as doi,
    ref.pmid as pmid,
    ref.arxiv as arxiv,
    ref.title as title,
    ref.authors as authors,
    ref.year as year,
    ref.raw as raw,
    provenance,
    current_timestamp() as created_timestamp,
    current_timestamp() as updated_timestamp
  FROM references_exploded


### This run's lists (oxjob #2054, #835 B3 + B4)

Every location's current list, its hash, and what it needs tonight: filed (new or changed), moved (its location belongs
to another work now) or dropped (gone or emptied).

In [0]:
%sql
-- Every location's current list with a reference, one row per list: a location's record (native_id, native_id_namespace, provenance; a
-- DOI registered with both Crossref and DataCite is two lists). pmid counts only on a reference without a DOI: the DOI step below
-- overwrites pmid on DOI rows. The hash must stay identical to the start step in notebooks/maintenance/work_references_refresh.sql.
CREATE OR REPLACE TABLE openalex.works.work_reference_lists_current AS
SELECT
  native_id,
  native_id_namespace,
  provenance,
  work_id AS citing_work_id,
  CASE version WHEN 'publishedVersion' THEN 1 WHEN 'acceptedVersion' THEN 2 WHEN 'submittedVersion' THEN 3 ELSE 4 END AS version_rank,
  xxhash64(transform(references, r -> named_struct('doi', r.doi, 'pmid', CASE WHEN r.doi IS NULL THEN r.pmid END,
    'arxiv', r.arxiv, 'title', r.title, 'authors', r.authors, 'year', r.year, 'raw', r.raw))) AS list_hash
FROM openalex.works.locations_mapped
WHERE work_id IS NOT NULL AND native_id IS NOT NULL AND native_id_namespace IS NOT NULL AND provenance IS NOT NULL
  -- a list with nothing to match (only blank entries: 962,429 lists on 6 Nov 2026) is not filed
  AND exists(references, r -> coalesce(r.doi, r.pmid, r.arxiv, r.raw, r.title) IS NOT NULL)
-- a record listed twice (40-45 a night on 3-5 Nov 2026, always the same list) counts once: its lowest work id, then
-- its lowest fingerprint
QUALIFY row_number() OVER (PARTITION BY native_id, native_id_namespace, provenance ORDER BY work_id, list_hash) = 1

In [0]:
%sql
-- One row per filed list: the fingerprint it was filed with and the work it is filed under. Started once by the Work
-- References Refresh job (oxjob #2054); kept by the cell after the matching steps.
CREATE TABLE IF NOT EXISTS openalex.works.work_reference_lists (
  native_id STRING,
  native_id_namespace STRING,
  provenance STRING,
  citing_work_id BIGINT,
  list_hash BIGINT,
  filed_at TIMESTAMP
) CLUSTER BY (native_id)

In [0]:
%sql
-- file: never filed, its contents changed, or its record now belongs to another work (its rows are filed again under
-- that work); drop: the record is gone or its list has no reference now. Unchanged lists don't appear.
CREATE OR REPLACE TABLE openalex.works.work_reference_lists_changes AS
SELECT
  coalesce(t.native_id, s.native_id) AS native_id,
  coalesce(t.native_id_namespace, s.native_id_namespace) AS native_id_namespace,
  coalesce(t.provenance, s.provenance) AS provenance,
  t.citing_work_id,
  t.list_hash,
  CASE WHEN t.native_id IS NULL THEN 'drop' ELSE 'file' END AS action
FROM openalex.works.work_reference_lists_current t
FULL OUTER JOIN openalex.works.work_reference_lists s
  ON s.native_id = t.native_id AND s.native_id_namespace = t.native_id_namespace AND s.provenance = t.provenance
WHERE t.native_id IS NULL OR s.native_id IS NULL OR NOT (s.list_hash <=> t.list_hash)
  OR NOT (s.citing_work_id <=> t.citing_work_id)

In [0]:
%sql
-- Tripwires: an empty list table would re-file every list; more than 5% of lists dropped, or more than 20% filed, in one
-- night means locations_mapped is broken or the fingerprint changed, not that publishers changed their lists (a normal
-- night files ~0.07% and drops ~0.0004%; the one-off refresh files ~16% and drops ~1.9%). Moves count as files, so a work-id mix-up stops the night too.
SELECT CASE
  WHEN (SELECT count(*) FROM openalex.works.work_reference_lists) = 0
   AND (SELECT count(*) FROM (SELECT 1 FROM openalex.works.work_references LIMIT 1)) > 0
  THEN raise_error('openalex.works.work_reference_lists is empty: run the Work References Refresh job, mode=start (oxjob #2054), first')
  WHEN (SELECT count(*) FROM openalex.works.work_reference_lists) > 0
   AND (SELECT count_if(action = 'drop') FROM openalex.works.work_reference_lists_changes)
     > 0.05 * (SELECT count(*) FROM openalex.works.work_reference_lists)
  THEN raise_error('over 5% of filed reference lists would be dropped tonight: check locations_mapped (oxjob #2054)')
  WHEN (SELECT count(*) FROM openalex.works.work_reference_lists) > 0
   AND (SELECT count_if(action = 'file') FROM openalex.works.work_reference_lists_changes)
     > 0.2 * (SELECT count(*) FROM openalex.works.work_reference_lists)
  THEN raise_error('over 20% of reference lists would be re-filed tonight: check locations_mapped and the fingerprint (oxjob #2054)')
END AS tripwire

In [0]:
%sql
-- Kept until the lists are recorded (cell after the matching steps), so a retry after a failed attempt still has the
-- matches of lists whose old rows that attempt already removed.
CREATE TABLE IF NOT EXISTS openalex.works.work_reference_kept_matches (
  native_id STRING,
  native_id_namespace STRING,
  provenance STRING,
  text_key STRING,
  cited_work_id BIGINT
)

In [0]:
%sql
-- Matches of references with no DOI and no printed id (the 2025 bulk parse's title matches, and PMID matches the PMID
-- step no longer makes once the cited work has a DOI),
-- captured before their lists are re-filed and put back after the matching steps when the location still lists the same
-- text (#835 B3 + B4 sizing, 3 Nov 2026: the exact text carries 270.1M of 288.6M Crossref title matches). Text key: raw
-- text, else title, letters and digits only, 20 or more of them.
MERGE INTO openalex.works.work_reference_kept_matches AS target
USING (
  SELECT
    wr.native_id,
    wr.native_id_namespace,
    wr.provenance,
    lower(regexp_replace(coalesce(wr.raw, wr.title), '[^A-Za-z0-9]', '')) AS text_key,
    max(wr.cited_work_id) AS cited_work_id
  FROM openalex.works.work_references wr
  LEFT SEMI JOIN openalex.works.work_reference_lists_changes c
    ON c.native_id = wr.native_id AND c.native_id_namespace = wr.native_id_namespace AND c.provenance = wr.provenance
    AND c.action = 'file'
  WHERE wr.cited_work_id IS NOT NULL AND wr.doi IS NULL AND wr.parsed_doi IS NULL
    AND length(regexp_replace(coalesce(wr.raw, wr.title), '[^A-Za-z0-9]', '')) >= 20
  GROUP BY 1, 2, 3, 4
) AS source
ON target.native_id = source.native_id AND target.native_id_namespace = source.native_id_namespace
  AND target.provenance = source.provenance AND target.text_key = source.text_key
WHEN NOT MATCHED THEN INSERT *

In [0]:
%sql
-- Before their rows go: tonight's lists lose their recorded fingerprint, so if this run stops before the last cell, the
-- next run files them again (even if their record has meanwhile gone back to how it was recorded).
MERGE INTO openalex.works.work_reference_lists AS target
USING openalex.works.work_reference_lists_changes AS source
ON target.native_id = source.native_id AND target.native_id_namespace = source.native_id_namespace
  AND target.provenance = source.provenance
WHEN MATCHED THEN UPDATE SET target.list_hash = NULL

In [0]:
%sql
-- Rows of lists filed or dropped tonight go; the insert below files the current list.
MERGE INTO openalex.works.work_references AS target
USING openalex.works.work_reference_lists_changes AS source
ON target.native_id = source.native_id AND target.native_id_namespace = source.native_id_namespace
  AND target.provenance = source.provenance
WHEN MATCHED THEN DELETE

### File new and changed lists into `work_references`

In [0]:
%sql
-- File the lists marked 'file' above (new, changed, or on another work now; their old rows are gone), one row per
-- reference, under the work that owns the record (oxjob #2054). Every other list's rows stay as they are.

INSERT INTO openalex.works.work_references (
  native_id,
  native_id_namespace,
  citing_work_id,
  cited_work_id,
  ref_ind,
  doi,
  pmid,
  arxiv,
  title,
  normalized_title,
  authors,
  year,
  raw,
  parsed_doi,
  parsed_first_author,
  parsed_title,
  title_author,
  provenance,
  created_timestamp,
  updated_timestamp
)
WITH lists_to_file AS (
  SELECT lm.native_id, lm.native_id_namespace, lm.work_id, lm.provenance, lm.references
  FROM openalex.works.locations_mapped lm
  LEFT SEMI JOIN openalex.works.work_reference_lists_changes c
    ON c.native_id = lm.native_id AND c.native_id_namespace = lm.native_id_namespace AND c.provenance = lm.provenance
    AND c.citing_work_id = lm.work_id AND c.action = 'file'
    AND c.list_hash = xxhash64(transform(references, r -> named_struct('doi', r.doi, 'pmid', CASE WHEN r.doi IS NULL THEN r.pmid END,
      'arxiv', r.arxiv, 'title', r.title, 'authors', r.authors, 'year', r.year, 'raw', r.raw)))
  WHERE lm.work_id IS NOT NULL AND exists(lm.references, r -> coalesce(r.doi, r.pmid, r.arxiv, r.raw, r.title) IS NOT NULL)
  -- the row the list cell fingerprinted; a record listed twice with the same list on one work is filed once
  QUALIFY row_number() OVER (PARTITION BY lm.native_id, lm.native_id_namespace, lm.provenance ORDER BY lm.work_id) = 1
), works_to_process AS (
  SELECT
    native_id,
    native_id_namespace,
    work_id as citing_work_id,
    provenance,
    posexplode(references) as (ref_ind, ref)
  FROM lists_to_file
)
-- [1972-extract begin] oxjob #1972: parsed_doi = the DOI (else the arXiv id, as 10.48550/arxiv.<id>) printed in the text of
-- a reference whose DOI field is empty. First DOI in the text; a DOI ending in . - / _ joined to a following token that
-- holds a digit (line-break splits: '10.1016/j. tips.2016.10.001'); trailing punctuation and an unbalanced ')' off;
-- funder DOIs (10.13039/) and '%' templates never taken. arXiv ids: new style after 'arXiv' / arxiv.org/abs, old style
-- archive/NNNNNNN (subject class dropped). The one-time backfill of older rows reuses this block (oxjob #1972 work/).
, x1 AS (
  SELECT *,
    CASE WHEN nullif(trim(ref.doi), '') IS NULL AND ref.raw IS NOT NULL
          AND (ref.raw RLIKE r'10\.[0-9]{4,9}(/|%2[fF])' OR ref.raw RLIKE r'(?i)arxiv'
               OR ref.raw RLIKE r'(?i)(hep-th|hep-ph|hep-ex|hep-lat|gr-qc|astro-ph|cond-mat|quant-ph|math-ph|nucl-th|nucl-ex|nlin|physics|math|cs|q-bio|chao-dyn|solv-int|alg-geom|dg-ga|funct-an|q-alg|cmp-lg|patt-sol|adap-org|chem-ph|supr-con|mtrl-th|atom-ph|acc-phys|plasm-ph|ao-sci|bayes-an|comp-gas)(\.[A-Za-z]{2})?/[0-9]{7}')
         THEN translate(replace(replace(lower(ref.raw), '%2f', '/'), '\u00ad', ''), '\u2010\u2011\u2012\u2013\u2014', '-----') END AS x_l
  FROM works_to_process
), x2 AS (
  SELECT *,
    regexp_extract(x_l, r'(10\.[0-9]{4,9}/[^\s"<>]+)(?:\s+([a-z0-9][^\s"<>]*))?', 1) AS x_g1,
    regexp_extract(x_l, r'(10\.[0-9]{4,9}/[^\s"<>]+)(?:\s+([a-z0-9][^\s"<>]*))?', 2) AS x_g2,
    regexp_extract(x_l, r'arxiv(?:\.org/(?:abs|pdf)/|\s*[:.]?\s*|\s+e-?prints?\s*:?\s*)([0-9]{4}\.[0-9]{4,5})', 1) AS x_axn,
    regexp_extract(x_l, r'(?:^|[^a-z-])(hep-th|hep-ph|hep-ex|hep-lat|gr-qc|astro-ph|cond-mat|quant-ph|math-ph|nucl-th|nucl-ex|nlin|physics|math|cs|q-bio|chao-dyn|solv-int|alg-geom|dg-ga|funct-an|q-alg|cmp-lg|patt-sol|adap-org|chem-ph|supr-con|mtrl-th|atom-ph|acc-phys|plasm-ph|ao-sci|bayes-an|comp-gas)(?:\.[a-z]{2})?/([0-9]{7})(?:[^0-9]|\z)', 1) AS x_axa,
    regexp_extract(x_l, r'(?:^|[^a-z-])(hep-th|hep-ph|hep-ex|hep-lat|gr-qc|astro-ph|cond-mat|quant-ph|math-ph|nucl-th|nucl-ex|nlin|physics|math|cs|q-bio|chao-dyn|solv-int|alg-geom|dg-ga|funct-an|q-alg|cmp-lg|patt-sol|adap-org|chem-ph|supr-con|mtrl-th|atom-ph|acc-phys|plasm-ph|ao-sci|bayes-an|comp-gas)(?:\.[a-z]{2})?/([0-9]{7})(?:[^0-9]|\z)', 2) AS x_axd
  FROM x1
), x3 AS (
  SELECT *, regexp_replace(CASE WHEN x_g1 RLIKE r'[.\-/_]\z' AND x_g2 RLIKE '[0-9]' THEN x_g1 || x_g2 ELSE x_g1 END,
    r'[.,;:\]\}\x27*]+\z', '') AS x_d1
  FROM x2
), x4 AS (
  SELECT *, CASE WHEN x_d1 LIKE '%)' AND length(x_d1) - length(replace(x_d1, '(', '')) < length(x_d1) - length(replace(x_d1, ')', ''))
         THEN regexp_replace(substr(x_d1, 1, length(x_d1) - 1), r'[.,;:\]]+\z', '') ELSE x_d1 END AS x_doi
  FROM x3
), x5 AS (
  SELECT *,
    CASE WHEN x_doi <> '' AND x_doi NOT LIKE '10.13039/%' AND x_doi NOT LIKE '%\%%' THEN x_doi
         WHEN x_axn <> '' THEN '10.48550/arxiv.' || x_axn
         WHEN x_axa <> '' THEN '10.48550/arxiv.' || x_axa || '/' || x_axd END AS x_parsed_doi
  FROM x4
)
-- [1972-extract end]
SELECT 
  native_id,
  native_id_namespace,
  citing_work_id,
  CAST(null as BIGINT), -- cited_work_id (calculated later)
  ref_ind,
  ref.doi as doi,
  ref.pmid as pmid,
  ref.arxiv as arxiv,
  ref.title as title,
  CAST(null as STRING), -- normalized_title (calculated later)
  ref.authors as authors,
  ref.year as year,
  ref.raw as raw,
  x_parsed_doi, -- parsed_doi: text DOI / arXiv id (oxjob #1972)
  CAST(null as STRING), -- parsed_first_author (calculated later)
  CAST(null as STRING), -- parsed_title (calculated later)
  CAST(null as STRING), -- title_author (calculated later)
  provenance,
  current_timestamp() as created_timestamp,
  current_timestamp() as updated_timestamp
FROM x5

### Live ids of `work_id_map` (oxjob #2054)

In [0]:
%sql
-- Every matching step below matches to live works only: on 6 Nov 2026 the lowest id of 1,178,670 DOIs in work_id_map
-- was a deleted work.
CREATE OR REPLACE TABLE openalex.works.work_id_map_live AS
SELECT m.* FROM openalex.works.work_id_map m
LEFT SEMI JOIN openalex.works.openalex_works_base b ON b.id = m.id

### Merge with `work_id_map.doi`

In [0]:
%sql
-- ============================================================
-- STEP 2: Link references to cited works via DOI matching
-- ============================================================
-- Update cited_work_id and title_author for references where DOI matches work_id_map
-- work_id_map.id is the single verdict column (oxjob #764 single-id migration); live ids only (work_id_map_live)
-- Only updates records where cited_work_id is still NULL
-- 10/23 added 29,913,749 IDs for 84,848,991 new records, recovered 39,496 pmid

MERGE INTO openalex.works.work_references AS target
USING (
  SELECT 
    LOWER(doi) as doi,
    MIN(id) as work_id,
    MIN(pmid) as pmid,
    MAX(title_author) as title_author 
  FROM openalex.works.work_id_map_live
  WHERE doi IS NOT NULL
  GROUP BY lower(doi)
) AS source
ON lower(target.doi) = source.doi
WHEN MATCHED AND target.cited_work_id IS NULL
THEN UPDATE SET
  target.cited_work_id = source.work_id,
  target.pmid = COALESCE(source.pmid, target.pmid), -- bring it in if exists
  target.title_author = COALESCE(source.title_author, target.title_author), -- bring it in if exists
  target.updated_timestamp = current_timestamp();

### Merge text DOIs / arXiv ids (`parsed_doi`) with `work_id_map` (oxjob #1972)

In [ ]:
%sql
-- oxjob #1972 (2 Nov 2026): a reference whose DOI field is empty but whose text prints a DOI or an arXiv id. parsed_doi is
-- filled at insert (cell above; older rows by a one-time backfill). Matched to work_id_map.doi (the lowest live id, as STEP 2), arXiv
-- ids also to work_id_map.arxiv; a joined line-break DOI that misses falls back to the first token alone.
-- Guards (the cited work's row in openalex_works_base, built just before this task):
--   * paratext cited work (an issue-level record a truncated or issue DOI lands on);
--   * reference text over 1,000 characters (page text, abstract books);
--   * contradiction: the reference has >= 4 Latin words of 5+ letters outside its URLs/DOIs, and shares no title word
--     (5+ letters) and no first-author name token (4+ letters) with the cited work; undecidable (no title / no author
--     name) counts as a contradiction;
--   * self-citation after merge redirects (page footers print the citing paper's own DOI).
-- Sized 2 Nov 2026: 36.0M new citations on 3.77M citing works; fresh 200-link sample 199/200 the reference's own id.
MERGE INTO openalex.works.work_references AS target
USING (
  WITH cand AS (
    SELECT citing_work_id, native_id, provenance, ref_ind, parsed_doi, raw,
      translate(replace(replace(lower(raw), '%2f', '/'), '\u00ad', ''), '\u2010\u2011\u2012\u2013\u2014', '-----') AS l
    FROM openalex.works.work_references
    WHERE cited_work_id IS NULL AND parsed_doi IS NOT NULL AND (doi IS NULL OR trim(doi) = '') AND citing_work_id IS NOT NULL
  ), cand2 AS (
    SELECT *, regexp_replace(regexp_extract(l, r'(10\.[0-9]{4,9}/[^\s"<>]+)', 1), r'[.,;:\]\}\x27*]+\z', '') AS p1 FROM cand
  ), cand3 AS (
    SELECT *, CASE WHEN p1 LIKE '%)' AND length(p1) - length(replace(p1, '(', '')) < length(p1) - length(replace(p1, ')', ''))
         THEN regexp_replace(substr(p1, 1, length(p1) - 1), r'[.,;:\]]+\z', '') ELSE p1 END AS plain_doi FROM cand2
  ), dmap AS (
    SELECT lower(doi) AS d, MIN(id) AS id FROM openalex.works.work_id_map_live WHERE doi IS NOT NULL GROUP BY lower(doi)
  ), amap AS (
    SELECT 'arxiv:' || lower(substr(arxiv, 7)) AS a, MIN(id) AS id FROM openalex.works.work_id_map_live
    WHERE lower(arxiv) LIKE 'arxiv:%' GROUP BY 1
  ), hit AS (
    SELECT c.*, COALESCE(d.id, a.id, dp.id) AS work_id
    FROM cand3 c
    LEFT JOIN dmap d ON d.d = c.parsed_doi
    LEFT JOIN amap a ON c.parsed_doi LIKE '10.48550/arxiv.%' AND a.a = 'arxiv:' || substr(c.parsed_doi, 16)
    LEFT JOIN dmap dp ON c.plain_doi <> c.parsed_doi AND dp.d = c.plain_doi
    WHERE COALESCE(d.id, a.id, dp.id) IS NOT NULL
  ), m AS (
    SELECT loser_work_id AS k, MAX_BY(winner_work_id, merged_at) AS v FROM openalex.works.merged_work_ids GROUP BY loser_work_id
  ), redirect AS (
    SELECT m1.k, COALESCE(m3.v, m2.v, m1.v) AS v FROM m m1 LEFT JOIN m m2 ON m2.k = m1.v LEFT JOIN m m3 ON m3.k = m2.v
  ), wb AS (
    SELECT id, type, title, get(authorships, 0).raw_author_name AS fa FROM openalex.works.openalex_works_base
    WHERE id IN (SELECT work_id FROM hit)
  ), g AS (
    SELECT h.citing_work_id, h.native_id, h.provenance, h.ref_ind, h.parsed_doi, h.work_id,
      COALESCE(rd.v, h.work_id) = COALESCE(rc.v, h.citing_work_id) AS is_self,
      coalesce(wb.type = 'paratext', false) AS is_paratext,
      length(h.raw) > 1000 AS is_long,
      filter(split(regexp_replace(translate(lower(h.raw), 'áàâäãåçéèêëíìîïñóòôöõøúùûüýÿşğıčšžłńśźżőűæœ', 'aaaaaaceeeeiiiinoooooouuuuyysgicszlnszzouae'),
        r'(https?://\S+|doi:\s*\S+|10\.[0-9]{4,9}/\S+|arxiv\S*)', ' '), '[^a-z]+'), w -> length(w) >= 4) AS rw4,
      filter(split(translate(lower(wb.title), 'áàâäãåçéèêëíìîïñóòôöõøúùûüýÿşğıčšžłńśźżőűæœ', 'aaaaaaceeeeiiiinoooooouuuuyysgicszlnszzouae'), '[^a-z]+'), w -> length(w) >= 5) AS tw,
      filter(split(translate(lower(wb.fa), 'áàâäãåçéèêëíìîïñóòôöõøúùûüýÿşğıčšžłńśźżőűæœ', 'aaaaaaceeeeiiiinoooooouuuuyysgicszlnszzouae'), '[^a-z]+'), w -> length(w) >= 4) AS faw
    FROM hit h
    LEFT JOIN redirect rd ON rd.k = h.work_id
    LEFT JOIN redirect rc ON rc.k = h.citing_work_id
    LEFT JOIN wb ON wb.id = h.work_id
  )
  SELECT citing_work_id, native_id, provenance, ref_ind, parsed_doi, MIN(work_id) AS work_id
  FROM g
  WHERE NOT is_self AND NOT is_paratext AND NOT is_long
    AND NOT coalesce(
      size(filter(rw4, w -> length(w) >= 5 AND NOT array_contains(array('journal','review','research','international',
        'proceedings','conference','science','sciences','university','available','retrieved','accessed','online','press',
        'studies','edition','volume','preprint','letters','https'), w))) >= 4
      AND size(tw) >= 2
      AND size(array_intersect(rw4, tw)) = 0
      AND size(array_intersect(rw4, faw)) = 0, true)
  GROUP BY citing_work_id, native_id, provenance, ref_ind, parsed_doi
) AS source
-- parsed_doi is part of the key: ~64K rows share (citing_work_id, ref_ind, provenance, native_id) with a different reference.
ON target.citing_work_id = source.citing_work_id AND target.ref_ind = source.ref_ind
  AND target.provenance = source.provenance AND target.native_id <=> source.native_id
  AND target.parsed_doi = source.parsed_doi
WHEN MATCHED AND target.cited_work_id IS NULL THEN UPDATE SET
  target.cited_work_id = source.work_id,
  target.updated_timestamp = current_timestamp();

### Merge with `work_id_map.pmid`

In [0]:
%sql
-- 10/23 added 11,311,131 IDs (where doi was null) for 84,848,991 new records 
MERGE INTO openalex.works.work_references AS target
USING (
  SELECT DISTINCT
    lower(pmid) as pmid,
    MIN(id) as work_id,
    MAX(title_author) as title_author
  FROM openalex.works.work_id_map_live
  WHERE pmid IS NOT NULL and doi is NULL -- keep doi is null because it otherwise adds a lot of erroneous refs
  GROUP BY pmid
) as source
ON lower(target.pmid) = source.pmid
WHEN MATCHED AND target.cited_work_id IS NULL
THEN UPDATE SET
  target.cited_work_id = source.work_id,
  target.title_author = COALESCE(source.title_author, target.title_author), -- bring it in if exists
  target.updated_timestamp = current_timestamp();

### Merge with `work_id_map.title_author`

In [0]:
%sql
-- 10/23 added 11,311,131 IDs for 84,848,991 new records
MERGE INTO openalex.works.work_references AS target
USING (
  SELECT DISTINCT
    lower(pmid) as pmid,
    MIN(id) as work_id,
    MAX(title_author) as title_author
  FROM openalex.works.work_id_map_live
  WHERE pmid IS NOT NULL AND doi is NULL
  GROUP BY pmid
) as source
ON lower(target.pmid) = source.pmid
WHEN MATCHED AND target.cited_work_id IS NULL
THEN UPDATE SET
  target.cited_work_id = source.work_id,
  target.title_author = COALESCE(source.title_author, target.title_author), -- bring it in if exists
  target.updated_timestamp = current_timestamp();

### Carry matches onto re-filed lists, record what was filed (oxjob #2054)

In [0]:
%sql
-- A reference with no DOI that no step above matched keeps the work it was matched to before its list was re-filed, when
-- the record still lists the same text, that work is live, and it is not the citing work.
MERGE INTO openalex.works.work_references AS target
USING (
  WITH filed AS (
    SELECT wr.native_id, wr.native_id_namespace, wr.provenance, wr.ref_ind, wr.citing_work_id,
      lower(regexp_replace(coalesce(wr.raw, wr.title), '[^A-Za-z0-9]', '')) AS text_key
    FROM openalex.works.work_references wr
    LEFT SEMI JOIN openalex.works.work_reference_lists_changes c
      ON c.native_id = wr.native_id AND c.native_id_namespace = wr.native_id_namespace AND c.provenance = wr.provenance
      AND c.action = 'file'
    WHERE wr.cited_work_id IS NULL AND wr.doi IS NULL
  )
  SELECT f.native_id, f.native_id_namespace, f.provenance, f.ref_ind, max(k.cited_work_id) AS cited_work_id
  FROM filed f
  JOIN openalex.works.work_reference_kept_matches k
    ON k.native_id = f.native_id AND k.native_id_namespace = f.native_id_namespace AND k.provenance = f.provenance
    AND k.text_key = f.text_key AND k.cited_work_id <> f.citing_work_id
  LEFT SEMI JOIN openalex.works.openalex_works_base b ON b.id = k.cited_work_id
  GROUP BY 1, 2, 3, 4
) AS source
ON target.native_id = source.native_id AND target.native_id_namespace = source.native_id_namespace
  AND target.provenance = source.provenance AND target.ref_ind = source.ref_ind
WHEN MATCHED AND target.cited_work_id IS NULL THEN UPDATE SET
  target.cited_work_id = source.cited_work_id,
  target.updated_timestamp = current_timestamp()

In [0]:
%sql
-- Record tonight's lists with their fingerprints (cleared above, before their rows went).
MERGE INTO openalex.works.work_reference_lists AS target
USING openalex.works.work_reference_lists_changes AS source
ON target.native_id = source.native_id AND target.native_id_namespace = source.native_id_namespace
  AND target.provenance = source.provenance
WHEN MATCHED AND source.action = 'drop' THEN DELETE
WHEN MATCHED THEN UPDATE SET
  target.citing_work_id = source.citing_work_id,
  target.list_hash = source.list_hash,
  target.filed_at = current_timestamp()
WHEN NOT MATCHED AND source.action <> 'drop' THEN INSERT (native_id, native_id_namespace, provenance, citing_work_id, list_hash,
  filed_at)
  VALUES (source.native_id, source.native_id_namespace, source.provenance, source.citing_work_id, source.list_hash,
  current_timestamp())

In [0]:
%sql
-- The kept matches have been put back and tonight's lists recorded.
TRUNCATE TABLE openalex.works.work_reference_kept_matches

In [0]:
%sql
  SELECT provenance, native_id, native_id_namespace, references
  FROM openalex.works.locations_parsed
  WHERE references is not null and size(references) > 0
  AND NOT(size(references) = 1 and references[0].doi is null 
    and references[0].pmid is null and references[0].title is null and references[0].arxiv is null and references[0].raw is null)

In [0]:
%sql
WITH filtered_refs AS (
  SELECT provenance, native_id, native_id_namespace, 
    references,
    size(array_compact(references.doi)) as num_dois,
    size(array_compact(references.pmid)) as num_pmids,
    size(array_compact(references.title)) as num_titles,
    size(array_compact(references.raw)) as num_raw,
    size(array_compact(references.authors)) as num_authors
  FROM openalex.works.locations_parsed
  WHERE references is not null and size(references) > 0
  AND NOT(size(references) = 1 and references[0].doi is null 
    and references[0].pmid is null and references[0].title is null 
    and references[0].arxiv is null and references[0].raw is null)
)
SELECT provenance,
  format_number(count(*),0) as total_record_count,
  format_number(sum(size(references)),0) as total_reference_count,
  format_number(sum(num_dois),0) as total_dois, 
  format_number(sum(num_pmids),0) as total_pmids, 
  format_number(sum(num_titles),0) as total_titles,   
  format_number(sum(num_authors),0) as total_authors,
  format_number(sum(num_raw),0) as total_raw_strings,  
  format_number(count_if(num_dois > 0),0) as has_any_dois,
  format_number(count_if(num_pmids > 0),0) as has_any_pmids, 
  format_number(count_if(num_titles > 0),0) as has_any_titles,  
  format_number(count_if(num_authors > 0),0) as has_any_authors,
  format_number(count_if(num_raw > 0),0) as has_any_raw_strings
FROM filtered_refs group by provenance order by 2 desc

### Aggregate before `openalex_works`

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.works.referenced_works
SELECT
  citing_work_id,
  ARRAY_SORT(COLLECT_SET(cited_work_id)) AS referenced_works
FROM openalex.works.work_references
WHERE cited_work_id IS NOT NULL
GROUP BY citing_work_id;
OPTIMIZE openalex.works.referenced_works ZORDER BY (citing_work_id);